In [3]:
import pandas as pd 
import numpy as np  
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from io import StringIO
import missingno as msno
pd.set_option("display.max_columns",None) 

In [4]:
df = pd.read_csv(r"C:\Users\HP\Desktop\playground-series-s5e8\playground-series-s5e8\dataset\train.csv")
X_test = pd.read_csv(r"C:\Users\HP\Desktop\playground-series-s5e8\playground-series-s5e8\dataset\test.csv")

In [11]:
y_train = df['y']
X_train = df.drop(columns=['y'])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from lightgbm import LGBMClassifier

# Feature lists
categorical_cols = ['job', 'marital', 'education', 'default',
                    'housing', 'loan', 'contact', 'month', 'poutcome']
binary_cols      = ['default', 'housing', 'loan']
cyclical_cols    = ['month']
ohe_cols         = ['job', 'marital', 'education', 'contact', 'poutcome']
numerical_cols   = ['age', 'balance', 'day', 'duration',
                    'campaign', 'pdays', 'previous']

# Assume X_train, y_train, X_test exist and X_train/X_test include 'id'
# Separate out test IDs
test_ids = X_test['id']
X_train = X_train.drop(columns=['id'])
X_test  = X_test.drop(columns=['id'])

# 1. FunctionTransformers
def clean_pdays(df):
    df = df.copy()
    df['pdays'] = df['pdays'].replace(-1, 0)
    return df

def shift_log(df):
    df = df.copy()
    for col in numerical_cols:
        min_val = df[col].min()
        shift = 1 - min_val if min_val <= 0 else 0
        df[col] = np.log1p(df[col] + shift)
    return df

def map_binary(df):
    df = df.copy()
    for c in binary_cols:
        df[c] = df[c].map({'no': 0, 'yes': 1})
    return df

def encode_month_cyclical(df):
    df = df.copy()
    mapping = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
               'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
    m = df['month'].map(mapping)
    df['month_sin'] = np.sin(2 * np.pi * m / 12)
    df['month_cos'] = np.cos(2 * np.pi * m / 12)
    return df[['month_sin','month_cos']]

from sklearn.preprocessing import FunctionTransformer
pdays_clean_tf    = FunctionTransformer(clean_pdays, validate=False)
shift_log_tf      = FunctionTransformer(shift_log, validate=False)
binary_map_tf     = FunctionTransformer(map_binary, validate=False)
month_cyclical_tf = FunctionTransformer(encode_month_cyclical, validate=False)

# 2. Numeric pipeline
numeric_pipeline = Pipeline([
    ('pdays_clean', pdays_clean_tf),
    ('shift_log',   shift_log_tf),
    ('scaler',      StandardScaler())
])

# 3. Categorical pipeline
categorical_pipeline = ColumnTransformer([
    ('binary', binary_map_tf,                     binary_cols),
    ('month',  month_cyclical_tf,                 cyclical_cols),
    ('ohe',    OneHotEncoder(handle_unknown='ignore'), ohe_cols)
], remainder='drop')

# 4. Full preprocessor
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,     numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
], remainder='drop')

# 5. LightGBM pipeline
pipeline_lgb = Pipeline([
    ('preprocess', preprocessor),
    ('clf',        LGBMClassifier(random_state=42))
])

# 6. Hyperparameter search space
param_dist = {
    'clf__n_estimators':   [100, 300, 500],
    'clf__learning_rate':  [0.01, 0.05, 0.1],
    'clf__num_leaves':     [31, 63, 127],
    'clf__subsample':      [0.6, 0.8, 1.0]
}

# 7. RandomizedSearchCV
rand_search = RandomizedSearchCV(
    pipeline_lgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 8. Fit on training data
rand_search.fit(X_train, y_train)

print("Best CV ROC-AUC:", rand_search.best_score_.round(4))
print("Best params:", rand_search.best_params_)

# 9. Final predictions on X_test
best_model = rand_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# 10. Prepare submission
submission = pd.DataFrame({'id': test_ids, 'y': y_pred_proba})
submission.to_csv('submission.csv', index=False)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Number of positive: 90488, number of negative: 659512
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057993 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1037
[LightGBM] [Info] Number of data points in the train set: 750000, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120651 -> initscore=-1.986283
[LightGBM] [Info] Start training from score -1.986283
Best CV ROC-AUC: 0.9679
Best params: {'clf__subsample': 0.8, 'clf__num_leaves': 63, 'clf__n_estimators': 500, 'clf__learning_rate': 0.05}
